In [ ]:
import os
import torch
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import (
    SegformerImageProcessor, 
    SegformerForSemanticSegmentation, 
    TrainingArguments, 
    Trainer,
    EarlyStoppingCallback
)

In [ ]:

IMAGE_DIR = "./data/JPEGImages"
MASK_DIR = "./data/SegmentationClass"
MODEL_ID = "nvidia/mit-b0"  # Small, fast, and accurate for most tasks

ID2LABEL = {0: "background", 1: "filth"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

In [ ]:
all_filenames = sorted([
    f for f in os.listdir(IMAGE_DIR) 
    if f.lower().endswith(('.jpg', '.png', '.jpeg'))
])

train_files, val_files = train_test_split(
    all_filenames, 
    test_size=0.2, 
    random_state=42, 
    shuffle=True
)

print(f"Total images: {len(all_filenames)}")
print(f"Training on: {len(train_files)}")
print(f"Validating on: {len(val_files)}")

In [ ]:
def run_sanity_check(filenames, img_dir, mask_dir):
    print(f"Starting check on {len(filenames)} files...")
    issues = 0
    
    for fname in filenames:
        img_path = os.path.join(img_dir, fname)
        mask_name = os.path.splitext(fname)[0] + ".png"
        mask_path = os.path.join(mask_dir, mask_name)
        
        if not os.path.exists(img_path):
            print(f"MISSING IMAGE: {img_path}")
            issues += 1
            continue
        if not os.path.exists(mask_path):
            print(f"MISSING MASK: {mask_path}")
            issues += 1
            continue
            
        try:
            img = Image.open(img_path)
            mask = Image.open(mask_path)
            
            if img.size != mask.size:
                print(f"SIZE MISMATCH: {fname} (Img: {img.size}, Mask: {mask.size})")
                issues += 1
            
            mask_array = np.array(mask)
            unique_vals = np.unique(mask_array)
            if len(unique_vals) <= 1:
                pass 
                
        except Exception as e:
            print(f"CORRUPT FILE: {fname} - {e}")
            issues += 1

    if issues == 0:
        print("Sanity check passed!")
    else:
        print(f"Found {issues} issues.")

run_sanity_check(all_filenames, IMAGE_DIR, MASK_DIR)

In [ ]:
class SegformerDataset(Dataset):
    def __init__(self, filenames, img_dir, mask_dir, processor):
        self.filenames = filenames
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.processor = processor

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        
        image = Image.open(os.path.join(self.img_dir, fname)).convert("RGB")
        mask_name = os.path.splitext(fname)[0] + ".png"
        mask = Image.open(os.path.join(self.mask_dir, mask_name)).convert("L")

        mask_array = np.array(mask)
        mask_array[mask_array > 0] = 1
        mask = Image.fromarray(mask_array)

        encoded_inputs = self.processor(
            image, 
            mask, 
            return_tensors="pt"
        )
        
        for k, v in encoded_inputs.items():
            encoded_inputs[k] = v.squeeze(0)
            
        return encoded_inputs

In [ ]:
processor = SegformerImageProcessor.from_pretrained(MODEL_ID)
processor.do_reduce_labels = False 

train_dataset = SegformerDataset(train_files, IMAGE_DIR, MASK_DIR, processor)
val_dataset = SegformerDataset(val_files, IMAGE_DIR, MASK_DIR, processor)

model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_ID,
    num_labels=len(ID2LABEL),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

In [ ]:
device = torch.device("cuda")
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Usage: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_ID,
    num_labels=len(ID2LABEL),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

model.to(device)

training_args = TrainingArguments(
    output_dir="./segformer-finetuned-results",
    learning_rate=6e-5,
    num_train_epochs=50,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    
    save_total_limit=2,
    logging_steps=10,
    fp16=True,
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

trainer.train()
trainer.save_model("./final_model")
processor.save_pretrained("./final_model")

In [ ]:
import torch
from torch import nn
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os

MODEL_DIR = "./final_model"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model from {MODEL_DIR}...")
print(f"Running on: {DEVICE}")

processor = SegformerImageProcessor.from_pretrained(MODEL_DIR)
print("✅ Loaded processor from ./final_model")


processor.do_reduce_labels = False 

model = SegformerForSemanticSegmentation.from_pretrained(MODEL_DIR)
model.to(DEVICE)
model.eval()

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    
    inputs = processor(images=image, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        

    logits = outputs.logits
    upsampled_logits = nn.functional.interpolate(
        logits,
        size=image.size[::-1],
        mode="bilinear",
        align_corners=False,
    )
    
    pred_seg = upsampled_logits.argmax(dim=1)[0]
    return image, pred_seg.cpu().numpy()

if not os.path.exists(IMAGE_PATH):
    print(f"Error: Image not found at {IMAGE_PATH}")
else:
    original, mask = predict(IMAGE_PATH)

    mask_overlay = np.zeros((mask.shape[0], mask.shape[1], 4))
    mask_overlay[mask == 1] = [1, 0, 0, 0.5]

    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(original)
    ax[0].set_title("Original Input")
    ax[0].axis("off")
    
    ax[1].imshow(original)
    ax[1].imshow(mask_overlay)
    ax[1].set_title("Prediction (Red = Filth)")
    ax[1].axis("off")
    
    plt.tight_layout()
    plt.show()